# HMR-GNN — Run on Kaggle (background, no tab-sitting)

**Heterophily-Aware Multi-Relational GNN for Robust Fake Account Detection.**

Unlike Colab, Kaggle can run this **in the background**: use **Save Version -> Save & Run All
(Commit)** and you can close the tab and multitask. The run finishes on Kaggle's servers and
the results are saved as the notebook's Output (downloadable).

### One-time setup (right panel -> Settings / Session options)
1. **Accelerator -> GPU T4 x2** (or P100).
2. **Internet -> On**  (needed to `git clone`; requires a phone-verified Kaggle account).
3. Then click **Save Version -> Save & Run All (Commit)** to run it in the background.

Results land in `/kaggle/working/results` and are bundled into `hmrgnn_results.zip`.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())

## 2. Clone the repository (code + data)
Requires **Internet -> On** in the session settings.

In [ ]:
import os
os.chdir('/kaggle/working')
if not os.path.isdir('/kaggle/working/HMR-GNN'):
    !git clone https://github.com/Vincentfernandes17/HMR-GNN.git
os.chdir('/kaggle/working/HMR-GNN')
!git pull
assert os.path.exists('data/MGTAB/features.pt'), 'MGTAB data missing - is Internet enabled for git clone?'
print('Repo ready. Data files:', os.listdir('data/MGTAB'))

## 3. Install dependencies
Kaggle ships a GPU build of PyTorch, so we only add the rest.

In [ ]:
!pip install -q "scikit-learn>=1.3" "scipy>=1.10" "pandas>=2.0" "numpy>=1.24"
import sklearn, scipy, pandas
print('sklearn', sklearn.__version__, '| scipy', scipy.__version__, '| pandas', pandas.__version__)

## 4. Quick smoke test (about 1 minute)

In [ ]:
import warnings; warnings.filterwarnings('ignore')
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python src/run_experiments.py --mode smoke --task bot --epochs 2 --patience 2 --quiet
print('\nSmoke test finished OK.')

## 5. Full experiment suite
Baselines + hyperparameter tuning + ablation on the bot task across 5 seeds. On a Kaggle
GPU this typically runs in well under an hour and fits comfortably in the background commit.

In [ ]:
OUTDIR = '/kaggle/working/results'
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python src/run_experiments.py \
    --mode all \
    --task bot \
    --seeds 42,52,62,72,82 \
    --epochs 500 \
    --patience 50 \
    --trials 12 \
    --output-dir {OUTDIR} \
    --quiet
print('\nFull suite complete.')

## 5b. Robustness to adversarial / spurious edges

Injects **label-independent random edges** (spurious follows / link-farming noise) at
0 / 5 / 10 / 20% of |E| and measures how far each model's macro F1 drops. The edge-gate in
HMR is meant to suppress uninformative edges, so HMR should degrade **least**; the MLP is a
graph-agnostic control and stays flat.

> This is the **leakage-free** setting. An earlier version injected cross-class-only edges,
> which leaked labels in this transductive graph (accuracy shot to ~100%); that mode was
> removed from evaluation. The cell below runs `git pull` first to ensure the fixed code.
>
> **Quick check:** change `--seeds 42,52,62` to `--seeds 42` for a ~15-minute single-seed run
> before committing to the full 3-seed version.

In [ ]:
# Robustness to spurious edges (leakage-free random injection).
# git pull grabs the latest fixed code. Use --seeds 42 for a ~15-min quick check.
!cd /kaggle/working/HMR-GNN && git pull && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
 python src/run_experiments.py --mode attack --task bot \
 --seeds 42,52,62 --epochs 500 --patience 50 \
 --attack-fractions 0.0,0.05,0.10,0.20 \
 --output-dir /kaggle/working/results --quiet
print('\nRobustness experiment complete.')

In [ ]:
import pandas as pd, os
OUTDIR = '/kaggle/working/results'
def show(path, title):
    if os.path.exists(path):
        print('\n===', title, '===')
        display(pd.read_csv(path))
    else:
        print('(missing)', path)
show(f'{OUTDIR}/bot/tables/baseline_comparison.csv', 'BASELINE COMPARISON (mean +/- std)')
show(f'{OUTDIR}/bot/tables/ablation_comparison.csv', 'ABLATION STUDY')
show(f'{OUTDIR}/bot/tables/significance_tests.csv', 'SIGNIFICANCE TESTS')
show(f'{OUTDIR}/bot/tables/hyperparameter_tuning.csv', 'HYPERPARAMETER TUNING')
show(f'{OUTDIR}/bot/tables/attack_robustness.csv', 'ADVERSARIAL ROBUSTNESS')
show(f'{OUTDIR}/bot/tables/attack_degradation.csv', 'ROBUSTNESS DEGRADATION (smaller drop = more robust)')

## 6. Download all results (zip + clickable link)
Finds the results directory wherever it ended up, zips it, and shows a download link.

In [ ]:
import os, glob, shutil

# Find the results directory wherever it ended up, then zip it.
candidates = ['/kaggle/working/results', '/kaggle/working/HMR-GNN/results', './results']
src = next((p for p in candidates if os.path.isdir(p) and os.listdir(p)), None)
if src is None:
    print('NO results found. CSVs under /kaggle/working:')
    print('\n'.join(glob.glob('/kaggle/working/**/tables/*.csv', recursive=True)) or 'nothing found')
else:
    print('Using results dir:', src)
    print('Files inside:', len(glob.glob(os.path.join(src, '**', '*.*'), recursive=True)))
    out = shutil.make_archive('/kaggle/working/hmrgnn_results', 'zip', src)
    print('Wrote:', out, '| size MB:', round(os.path.getsize(out) / 1e6, 2))
    from IPython.display import FileLink
    os.chdir('/kaggle/working')
    display(FileLink('hmrgnn_results.zip'))